# Gold Price Prediction System – Core Forecast Notebook

This notebook implements a **machine-learning-based gold price forecasting pipeline** using:
- **yfinance** – to download daily market data (gold, S&P 500, VIX, USD, oil, 10Y yield, BTC)
- **pandas / numpy** – for feature engineering and time-series manipulation
- **scikit-learn (Random Forest)** – for training and prediction

### What this notebook does, step by step
1. Downloads historical daily price data for several market assets.
2. Resamples data to monthly frequency and engineers meaningful features (momentum, volatility, lags, cyclical encodings, crisis flags).
3. Trains a Random Forest model on historical monthly data.
4. Generates a recursive multi-month forecast with 95 % confidence intervals.
5. Displays metrics, a forecast table, and a chart.

> **Beginner tip:** Run cells from top to bottom using *Shift + Enter*. All functions are defined before they are called.

## 1. Imports & Setup

We suppress noisy warnings and import all required libraries.
If a library is missing, install it with `pip install <library_name>`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import math
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("Libraries loaded successfully.")

## 2. Configuration

`DEFAULT_CONFIG` holds all tunable parameters in one place so you can easily adjust them without hunting through the code.

In [ ]:
DEFAULT_CONFIG = {
    "start_date": "2000-01-01",
    "end_date": None,          # None = today
    "random_state": 42,
    "n_estimators": 500,
    "max_depth": 8,
    "min_samples_leaf": 2,
    "use_manual_event_flags": True
}

print("Default config:", DEFAULT_CONFIG)

## 3. Data Download

The function below downloads daily closing prices for several market assets.
It tries a primary ticker first and falls back to an alternative if the primary fails (useful in cloud environments where some futures tickers may not be available).

In [ ]:
def download_market_data(start_date: str, end_date=None) -> pd.DataFrame:
    """Download daily close prices for gold and macro indicators.

    Each asset has one or more ticker candidates.  The first one that
    returns non-empty data is used; this makes the pipeline robust to
    cloud environments where some futures/index tickers are unavailable.
    """
    # Primary + fallback symbols for reliability on cloud environments
    ticker_candidates = {
        "gold":      ["GC=F", "GLD"],          # Gold futures -> fallback ETF
        "sp500":     ["^GSPC", "SPY"],          # Index -> fallback ETF
        "vix":       ["^VIX"],
        "usd_proxy": ["UUP", "DX-Y.NYB"],
        "oil":       ["CL=F", "BZ=F"],          # WTI -> fallback Brent
        "tnx":       ["^TNX"],
        "btc":       ["BTC-USD"]
    }

    def fetch_close_series(symbol: str):
        """Return a clean numeric Series of daily closes or None."""
        data = yf.download(symbol, start=start_date, end=end_date,
                           auto_adjust=True, progress=False)
        if data is None or data.empty or "Close" not in data.columns:
            return None

        close_data = data["Close"]

        # yfinance may return Series or DataFrame depending on version/options
        if isinstance(close_data, pd.DataFrame):
            if close_data.shape[1] == 0:
                return None
            s = close_data.iloc[:, 0]
        else:
            s = close_data

        s = pd.to_numeric(s, errors="coerce").dropna()
        if s.empty:
            return None
        return s

    frames = []
    used_symbols = {}

    for col_name, candidates in ticker_candidates.items():
        chosen_series = None
        for sym in candidates:
            s = fetch_close_series(sym)
            if s is not None and not s.empty:
                chosen_series = s.rename(col_name)
                used_symbols[col_name] = sym
                break

        if chosen_series is None:
            raise ValueError(
                f"Downloaded empty data for ticker group: {col_name} "
                f"(tried {candidates})"
            )

        frames.append(chosen_series)

    df = pd.concat(frames, axis=1).sort_index()
    df = df.ffill().dropna()

    if df.empty:
        raise ValueError("Final merged market dataset is empty after alignment.")

    return df


print("download_market_data defined.")

## 4. Feature Engineering

Raw daily prices are transformed into a **monthly feature matrix**.  Features include:
- Monthly averages of each asset price
- Momentum (month-over-month returns)
- Realised volatility (standard deviation of daily returns within the month)
- Rolling indicators (VIX fear flag, S&P drawdown, USD/rate trend)
- Lagged gold prices (1, 2, 3, 6, 12 months)
- Calendar features with cyclical encoding (sin/cos of month)
- Manual event flags for known macroeconomic shocks

In [ ]:
def apply_manual_event_flags(monthly_df: pd.DataFrame) -> pd.DataFrame:
    """Set binary flags for known historical macro shock periods."""
    m = monthly_df.copy()

    # Historical examples (editable)
    m.loc[(m.index >= "2020-03-01") & (m.index <= "2020-06-01"), "global_crisis_flag"] = 1
    m.loc[(m.index >= "2022-02-01") & (m.index <= "2023-12-01"), "war_shock_flag"] = 1
    m.loc[(m.index >= "2021-06-01") & (m.index <= "2023-06-01"), "inflation_shock_flag"] = 1

    return m


def make_monthly_features(daily_df: pd.DataFrame, use_manual_event_flags: bool = True) -> pd.DataFrame:
    """Resample daily data to monthly and engineer all model features."""
    df = daily_df.copy()

    # Daily returns (used for realised-volatility features)
    for c in ["gold", "sp500", "usd_proxy", "oil", "btc"]:
        df[f"{c}_ret_1d"] = df[c].pct_change()

    monthly = pd.DataFrame(index=df.resample("MS").mean().index)

    # Monthly averages
    for c in ["gold", "sp500", "vix", "usd_proxy", "oil", "tnx", "btc"]:
        monthly[f"{c}_avg"] = df[c].resample("MS").mean()

    # Momentum (month-over-month price change)
    for c in ["gold", "sp500", "usd_proxy", "oil", "btc"]:
        monthly[f"{c}_mom"] = monthly[f"{c}_avg"].pct_change()

    # Realised volatility (std of daily returns within month)
    for c in ["gold", "sp500", "usd_proxy", "oil", "btc"]:
        monthly[f"{c}_rv"] = df[f"{c}_ret_1d"].resample("MS").std()

    # VIX fear flag: 1 when VIX is >20 % above its 24-month median
    monthly["fear_flag"] = (
        monthly["vix_avg"] >= monthly["vix_avg"].rolling(24, min_periods=6).median() * 1.2
    ).astype(int)

    # S&P 500 drawdown from 12-month rolling high
    rolling_max = monthly["sp500_avg"].rolling(12, min_periods=3).max()
    monthly["sp500_drawdown"] = (monthly["sp500_avg"] / rolling_max) - 1.0

    # USD uptrend flag: 1 when current level > 6-month rolling mean
    usd_roll = monthly["usd_proxy_avg"].rolling(6, min_periods=3).mean()
    monthly["usd_uptrend"] = (monthly["usd_proxy_avg"] > usd_roll).astype(int)

    # 10Y yield trend flag
    tnx_roll = monthly["tnx_avg"].rolling(6, min_periods=3).mean()
    monthly["rates_uptrend"] = (monthly["tnx_avg"] > tnx_roll).astype(int)

    # Lagged gold price features
    for lag in [1, 2, 3, 6, 12]:
        monthly[f"gold_lag_{lag}"] = monthly["gold_avg"].shift(lag)

    # Target: next month's gold average price
    monthly["target_next_gold"] = monthly["gold_avg"].shift(-1)

    # Calendar features with cyclical encoding
    monthly["month"] = monthly.index.month
    monthly["quarter"] = monthly.index.quarter
    monthly["month_sin"] = np.sin(2 * np.pi * monthly["month"] / 12.0)
    monthly["month_cos"] = np.cos(2 * np.pi * monthly["month"] / 12.0)

    # Manual event flags (default 0, overwritten below for known periods)
    monthly["global_crisis_flag"] = 0
    monthly["war_shock_flag"] = 0
    monthly["inflation_shock_flag"] = 0

    if use_manual_event_flags:
        monthly = apply_manual_event_flags(monthly)

    monthly = monthly.dropna().copy()
    return monthly


def get_feature_columns():
    """Return the ordered list of feature column names used by the model."""
    return [
        "gold_avg", "sp500_avg", "vix_avg", "usd_proxy_avg", "oil_avg", "tnx_avg", "btc_avg",
        "gold_mom", "sp500_mom", "usd_proxy_mom", "oil_mom", "btc_mom",
        "gold_rv", "sp500_rv", "usd_proxy_rv", "oil_rv", "btc_rv",
        "fear_flag", "sp500_drawdown", "usd_uptrend", "rates_uptrend",
        "gold_lag_1", "gold_lag_2", "gold_lag_3", "gold_lag_6", "gold_lag_12",
        "month", "quarter", "month_sin", "month_cos",
        "global_crisis_flag", "war_shock_flag", "inflation_shock_flag"
    ]


print("Feature engineering functions defined.")

## 5. Model Training

A **Random Forest Regressor** is trained on 85 % of the monthly data; the remaining 15 % is used as a hold-out test set to compute evaluation metrics:
- **MAE** – Mean Absolute Error (in USD)
- **RMSE** – Root Mean Squared Error (in USD)
- **MAPE %** – Mean Absolute Percentage Error

In [ ]:
def train_model(monthly_df: pd.DataFrame, config: dict):
    """Train a Random Forest model and evaluate it on a hold-out test set.

    Returns
    -------
    model         : fitted RandomForestRegressor
    features      : list of feature column names
    metrics       : dict with MAE, RMSE, MAPE_%
    residual_std  : std of training residuals (used for confidence intervals)
    """
    features = get_feature_columns()

    X = monthly_df[features]
    y = monthly_df["target_next_gold"]

    # 85/15 chronological train/test split (no shuffling to avoid data leakage)
    split_idx = int(len(monthly_df) * 0.85)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    model = RandomForestRegressor(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        min_samples_leaf=config["min_samples_leaf"],
        random_state=config["random_state"],
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    pred_test = model.predict(X_test)
    mae  = mean_absolute_error(y_test, pred_test)
    rmse = math.sqrt(mean_squared_error(y_test, pred_test))
    mape = np.mean(np.abs((y_test - pred_test) / y_test)) * 100

    train_pred = model.predict(X_train)
    residual_std = np.std(y_train - train_pred)

    metrics = {"MAE": mae, "RMSE": rmse, "MAPE_%": mape}
    return model, features, metrics, residual_std


print("train_model defined.")

## 6. Forecasting

Three helper functions handle the forward-looking forecast:
1. **`build_future_dataframe`** – creates a skeleton DataFrame for future months, filling exogenous columns with the last observed values.
2. **`apply_scenario_to_future`** – applies a named scenario (*baseline*, *crisis*, *risk_on*) and/or custom percentage shocks to macro variables.
3. **`recursive_forecast`** – iterates month by month, feeding each predicted gold price back as a feature for the next step, and computing 95 % prediction intervals.

In [ ]:
def build_future_dataframe(last_monthly_row: pd.Series, forecast_start: str, periods: int) -> pd.DataFrame:
    """Create a future DataFrame pre-filled with last known exogenous values."""
    future_idx = pd.date_range(start=forecast_start, periods=int(periods), freq="MS")
    future = pd.DataFrame(index=future_idx)

    exog_cols = [
        "sp500_avg", "vix_avg", "usd_proxy_avg", "oil_avg", "tnx_avg", "btc_avg",
        "sp500_mom", "usd_proxy_mom", "oil_mom", "btc_mom",
        "sp500_rv", "usd_proxy_rv", "oil_rv", "btc_rv",
        "fear_flag", "sp500_drawdown", "usd_uptrend", "rates_uptrend",
        "global_crisis_flag", "war_shock_flag", "inflation_shock_flag"
    ]

    for c in exog_cols:
        future[c] = float(last_monthly_row[c])

    future["month"]     = future.index.month
    future["quarter"]   = future.index.quarter
    future["month_sin"] = np.sin(2 * np.pi * future["month"] / 12.0)
    future["month_cos"] = np.cos(2 * np.pi * future["month"] / 12.0)

    return future


def apply_scenario_to_future(
    future_df: pd.DataFrame,
    scenario,
    vix_bump_pct: float = 0.0,
    oil_bump_pct: float = 0.0,
    tnx_bump_pct: float = 0.0,
    usd_bump_pct: float = 0.0
) -> pd.DataFrame:
    """Apply a named scenario and/or custom macro shocks to the future DataFrame.

    Scenarios
    ---------
    'baseline'  – no change (default)
    'crisis'    – raise fear flags, increase VIX/oil
    'risk_on'   – lower fear flags, reduce VIX
    """
    f = future_df.copy()
    scenario_val = "{}".format(scenario).strip().lower()

    if scenario_val == "crisis":
        f["global_crisis_flag"]  = 1
        f["war_shock_flag"]      = 1
        f["inflation_shock_flag"] = 1
        f["fear_flag"]           = 1
        f["vix_avg"] = f["vix_avg"] * 1.20
        f["oil_avg"] = f["oil_avg"] * 1.10
    elif scenario_val == "risk_on":
        f["global_crisis_flag"]  = 0
        f["war_shock_flag"]      = 0
        f["inflation_shock_flag"] = 0
        f["fear_flag"]           = 0
        f["vix_avg"] = f["vix_avg"] * 0.90
    # else: baseline – no change

    # Apply any custom percentage bumps on top of the scenario
    f["vix_avg"]       = f["vix_avg"]       * (1.0 + float(vix_bump_pct) / 100.0)
    f["oil_avg"]       = f["oil_avg"]       * (1.0 + float(oil_bump_pct) / 100.0)
    f["tnx_avg"]       = f["tnx_avg"]       * (1.0 + float(tnx_bump_pct) / 100.0)
    f["usd_proxy_avg"] = f["usd_proxy_avg"] * (1.0 + float(usd_bump_pct) / 100.0)

    return f


def recursive_forecast(model, features, history_df: pd.DataFrame, future_df: pd.DataFrame, residual_std: float):
    """Step through future months one at a time, using each predicted gold
    price as input for the next step (recursive / auto-regressive forecast).

    Returns a DataFrame with columns:
      predicted_gold_avg, predicted_lower_95, predicted_upper_95
    """
    gold_series = history_df["gold_avg"].copy()
    preds, lower, upper = [], [], []

    for dt in future_df.index:
        row = future_df.loc[dt].copy()

        # Fill gold-derived features from the rolling history
        row["gold_avg"]  = gold_series.iloc[-1]
        row["gold_mom"]  = gold_series.pct_change().iloc[-1] if len(gold_series) > 1 else 0.0

        recent_changes   = gold_series.pct_change().dropna().tail(6)
        row["gold_rv"]   = recent_changes.std() if len(recent_changes) > 2 else 0.01

        for lag in [1, 2, 3, 6, 12]:
            row[f"gold_lag_{lag}"] = (
                gold_series.iloc[-lag] if len(gold_series) >= lag else gold_series.iloc[0]
            )

        x_row = pd.DataFrame([row])[features]
        yhat  = model.predict(x_row)[0]

        preds.append(float(yhat))
        lower.append(float(yhat - 1.96 * residual_std))
        upper.append(float(yhat + 1.96 * residual_std))

        # Feed predicted value back for next step
        gold_series.loc[dt] = yhat

    out = pd.DataFrame({
        "predicted_gold_avg":   preds,
        "predicted_lower_95":   lower,
        "predicted_upper_95":   upper
    }, index=future_df.index)

    return out


print("Forecasting functions defined.")

## 7. End-to-End Pipeline

`run_end_to_end` is the single entry-point that chains all the steps above.
Pass your desired forecast horizon, scenario, and optional macro shocks to get a complete result dictionary.

In [ ]:
def run_end_to_end(
    forecast_start: str,
    forecast_months: int,
    scenario="baseline",
    vix_bump_pct: float = 0.0,
    oil_bump_pct: float = 0.0,
    tnx_bump_pct: float = 0.0,
    usd_bump_pct: float = 0.0,
    config: dict = None
):
    """Run the full gold forecast pipeline and return a results dictionary.

    Parameters
    ----------
    forecast_start  : str   – first forecast month, e.g. '2026-05-01'
    forecast_months : int   – number of months to forecast
    scenario        : str   – 'baseline', 'crisis', or 'risk_on'
    vix_bump_pct    : float – % change applied to VIX (e.g. 10 = +10 %)
    oil_bump_pct    : float – % change applied to oil price
    tnx_bump_pct    : float – % change applied to 10Y Treasury yield
    usd_bump_pct    : float – % change applied to USD proxy
    config          : dict  – optional overrides for DEFAULT_CONFIG

    Returns
    -------
    dict with keys: 'daily', 'monthly', 'forecast', 'metrics', 'feature_importance'
    """
    cfg = DEFAULT_CONFIG.copy()
    if config:
        cfg.update(config)

    print("[1/5] Downloading market data...")
    daily = download_market_data(cfg["start_date"], cfg["end_date"])

    print("[2/5] Engineering monthly features...")
    monthly = make_monthly_features(daily, use_manual_event_flags=cfg["use_manual_event_flags"])

    print("[3/5] Training model...")
    model, features, metrics, residual_std = train_model(monthly, cfg)

    print("[4/5] Building future scenario...")
    future = build_future_dataframe(monthly.iloc[-1], forecast_start, int(forecast_months))
    future = apply_scenario_to_future(
        future,
        scenario=scenario,
        vix_bump_pct=vix_bump_pct,
        oil_bump_pct=oil_bump_pct,
        tnx_bump_pct=tnx_bump_pct,
        usd_bump_pct=usd_bump_pct
    )

    print("[5/5] Generating recursive forecast...")
    forecast = recursive_forecast(model, features, monthly, future, residual_std)

    fi = pd.DataFrame({
        "feature":    features,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)

    return {
        "daily":              daily,
        "monthly":            monthly,
        "forecast":           forecast,
        "metrics":            metrics,
        "feature_importance": fi
    }


print("run_end_to_end defined.")

## 8. Example Run

Run the complete pipeline and display:
1. Model evaluation metrics
2. Forecast table (predicted gold price with 95 % confidence interval)
3. Forecast chart overlaid on historical monthly gold prices
4. Top 10 most important features

> **Note:** The first run downloads ~25 years of daily data and trains the model, so it may take 1–3 minutes.

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
FORECAST_START  = "2026-05-01"   # First month to forecast
FORECAST_MONTHS = 20             # How many months ahead
SCENARIO        = "baseline"     # 'baseline', 'crisis', or 'risk_on'

# ── Run ───────────────────────────────────────────────────────────────────────
result = run_end_to_end(
    forecast_start=FORECAST_START,
    forecast_months=FORECAST_MONTHS,
    scenario=SCENARIO
)

forecast = result["forecast"]
monthly  = result["monthly"]
metrics  = result["metrics"]
fi       = result["feature_importance"]

# ── 1. Model metrics ──────────────────────────────────────────────────────────
print("\n=== Model Evaluation Metrics (hold-out test set) ===")
print(f"  MAE    : {metrics['MAE']:.2f} USD")
print(f"  RMSE   : {metrics['RMSE']:.2f} USD")
print(f"  MAPE % : {metrics['MAPE_%']:.2f} %")

# ── 2. Forecast table ─────────────────────────────────────────────────────────
print("\n=== Forecast Table ===")
display(forecast.round(2))

# ── 3. Forecast chart ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly.index, monthly["gold_avg"], label="Historical gold (monthly avg)", color="goldenrod")
ax.plot(forecast.index, forecast["predicted_gold_avg"], "--", label="Forecast", color="steelblue")
ax.fill_between(
    forecast.index,
    forecast["predicted_lower_95"],
    forecast["predicted_upper_95"],
    alpha=0.2, color="steelblue", label="95 % confidence interval"
)
ax.set_title("Gold Price Forecast")
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.legend()
plt.tight_layout()
plt.show()

# ── 4. Top feature importances ────────────────────────────────────────────────
print("\n=== Top 10 Feature Importances ===")
display(fi.head(10).reset_index(drop=True))